In [ ]:
from src.config import *
from src.tables_cleaner import clean_all_tables
from src.utils import load_tables, log_dfs

import os
import io

import openpyxl
import numpy as np
import pandas as pd

## __Data preprocessing__

In [ ]:
clean_all_tables()

In [ ]:
competitions_results = load_tables()

In [ ]:
log_dfs(competitions_results, 'competitions_results')

In [ ]:
competitions_results['FISU'].head()

__Type casting and calculating ages:__

In [ ]:
columns = competitions_results['FISU'].columns

for name, df in competitions_results.items():
    competitions_results[name].columns = columns
    competitions_results[name]['Rnk'] = df['Rnk'].astype('Int64')
    competitions_results[name]['Total'] = df['Total'].astype('float64')
    competitions_results[name]['IPF GL'] = df['IPF GL'].astype('float64')
    competitions_results[name]['Pts'] = df['Pts'].astype('Int64')

    if not pd.api.types.is_integer_dtype(competitions_results[name]['d.o.b.']):
        competitions_results[name]['d.o.b.'] = pd.to_datetime(df['d.o.b.']).dt.year

    competitions_results[name]['Age'] = COMPETITIONS_YEAR - competitions_results[name]['d.o.b.']

log_dfs(dfs=competitions_results, file_name='competitions_results_type_casted')

In [ ]:
competitions_results['FISU']

In [ ]:
competitions_results['Masters']['Age'].value_counts()

__Concatenate multiple competition tables into a single unified dataset__

In [ ]:
all_results = pd.concat(competitions_results.values(), axis=0)
all_results = all_results.reset_index().drop(columns='index')

all_results.info()

In [ ]:
all_results.head()

Calculating best attempts for each person

In [ ]:
for best_col, attempts in LIFTS.items():
    all_results[best_col] = all_results[attempts].max(axis=1)

all_results.head()

__Handle missing totals and failed attempts__

In [ ]:
all_results.info()

In [ ]:
all_results = all_results.dropna(subset=['Total'], axis=0, ignore_index=True)

In [ ]:
all_results.info()

In [ ]:
all_results.to_pickle(os.path.join(PROCESSED_PATH, "all_results.pkl"))